# NovaBank Digital Scam-To-Mule Baseline

This notebook introduces the first **NovaBank Digital** **Scam-to-mule flow** scenario. The task is to score alerts where a scam victim payment lands in an early-life account and is rapidly passed onward through a new beneficiary.

In [ ]:
import pandas as pd

from banking_fraud_lab import (
    build_learner_facing_views,
    evaluate_alert_scores,
    generate_digital_scam_to_mule_world,
    load_tables_to_sqlite,
)
from banking_fraud_lab.generators.digital_banking import DIGITAL_SCAM_TO_MULE_ACTIVITY_TYPE
from banking_fraud_lab.schema import PROTECTED_SCENARIO_ANSWER_KEYS

## Generate The Scenario

The generated world keeps **Client**, **User**, and **Partner** separate. Learner-facing views include telemetry, beneficiaries, alerts, cases, and outcomes, while protected answer keys remain outside the normal data view.

In [ ]:
tables = generate_digital_scam_to_mule_world(seed=42, scenario_prevalence=1.0)
learner_tables = build_learner_facing_views(tables)

assert PROTECTED_SCENARIO_ANSWER_KEYS in tables
assert PROTECTED_SCENARIO_ANSWER_KEYS not in learner_tables

pd.DataFrame(
    [
        {"table_name": table_name, "row_count": len(frame)}
        for table_name, frame in learner_tables.items()
    ]
)

## Build Scam-To-Mule Features

The feature query joins alerts to transactions, accounts, Users, Clients, Partners, session telemetry, beneficiary changes, and case outcomes. It also counts distinct Users per device fingerprint so shared-device behavior can be scored.

In [ ]:
connection = load_tables_to_sqlite(learner_tables, ":memory:")
digital_cases = pd.read_sql_query(
    """
    WITH device_user_counts AS (
        SELECT
            device_fingerprint_hash,
            COUNT(DISTINCT user_id) AS device_user_count
        FROM sessions
        GROUP BY device_fingerprint_hash
    )
    SELECT
        c.case_id,
        c.alert_id,
        al.alert_type,
        t.transaction_id,
        CAST(t.amount_chf AS REAL) AS amount_chf,
        JULIANDAY(t.booked_at) - JULIANDAY(a.opened_at) AS account_age_days,
        u.user_id,
        cl.client_id,
        p.partner_id,
        s.user_agent,
        s.app_or_browser_version,
        s.device_fingerprint_hash,
        s.ip_country,
        s.asn_risk_score,
        s.coarse_geolocation,
        s.is_vpn_or_proxy,
        s.auth_method,
        s.session_event,
        pb.beneficiary_change_event,
        duc.device_user_count,
        CAST(co.confirmed_fraud AS INTEGER) AS confirmed_fraud,
        CAST(co.loss_amount_chf AS REAL) AS loss_amount_chf
    FROM cases AS c
    JOIN alerts AS al
        ON c.alert_id = al.alert_id
    JOIN transactions AS t
        ON c.transaction_id = t.transaction_id
    JOIN accounts AS a
        ON c.account_id = a.account_id
    LEFT JOIN users AS u
        ON c.user_id = u.user_id
    LEFT JOIN clients AS cl
        ON u.client_id = cl.client_id
    LEFT JOIN partners AS p
        ON cl.partner_id = p.partner_id
    LEFT JOIN sessions AS s
        ON c.session_id = s.session_id
    LEFT JOIN payment_beneficiaries AS pb
        ON c.payment_beneficiary_id = pb.payment_beneficiary_id
    LEFT JOIN device_user_counts AS duc
        ON s.device_fingerprint_hash = duc.device_fingerprint_hash
    JOIN case_outcomes AS co
        ON c.case_id = co.case_id
    WHERE al.institution_name = 'NovaBank Digital'
    ORDER BY c.opened_at
    """,
    connection,
)

assert not digital_cases.empty
assert {"user_id", "client_id", "partner_id"}.issubset(digital_cases.columns)
assert digital_cases["confirmed_fraud"].nunique() == 2
digital_cases

## Score Alerts

This baseline uses a transparent scoring rule. It increases score for early-life accounts, new-beneficiary events, shared devices, high-risk network telemetry, and the scam-to-mule alert type.

In [ ]:
features = digital_cases.copy()
features["account_age_days"] = features["account_age_days"].fillna(999.0)
features["device_user_count"] = features["device_user_count"].fillna(1)
features["asn_risk_score"] = features["asn_risk_score"].fillna(0)
features["is_vpn_or_proxy"] = features["is_vpn_or_proxy"].fillna(0).astype(int)
features["score"] = (
    0.30 * (features["account_age_days"] <= 7).astype(float)
    + 0.25 * features["beneficiary_change_event"].eq("new_beneficiary_added").astype(float)
    + 0.20 * (features["device_user_count"] > 1).astype(float)
    + 0.15
    * ((features["asn_risk_score"] >= 80) | (features["is_vpn_or_proxy"] == 1)).astype(float)
    + 0.10 * features["alert_type"].eq(DIGITAL_SCAM_TO_MULE_ACTIVITY_TYPE).astype(float)
).clip(upper=1.0)

features[
    [
        "case_id",
        "alert_type",
        "account_age_days",
        "beneficiary_change_event",
        "device_user_count",
        "asn_risk_score",
        "is_vpn_or_proxy",
        "score",
        "confirmed_fraud",
    ]
]

## Evaluate Threshold Tradeoffs

Alert-aware metrics expose precision, recall, PR-AUC, alert volume, capacity, and cost tradeoffs. No headline accuracy is reported because fraud labels are sparse and case outcomes are operational decisions.

In [ ]:
report = evaluate_alert_scores(
    features[["case_id", "alert_id"]],
    features[["case_id", "confirmed_fraud", "loss_amount_chf"]],
    features[["alert_id", "score"]],
    thresholds=(0.35, 0.6, 0.8),
    alert_capacity=2,
    investigation_cost_chf=35.0,
    false_positive_cost_chf=10.0,
)

threshold_summary = pd.DataFrame(report["threshold_summaries"])
threshold_summary.insert(0, "pr_auc", report["pr_auc"])
assert "accuracy is out of scope" in report["limitation_summary"]
threshold_summary[
    [
        "pr_auc",
        "threshold",
        "precision",
        "recall",
        "alert_volume",
        "alert_capacity",
        "over_capacity_alerts",
        "total_cost_chf",
    ]
]

In [ ]:
connection.close()